## Ch15. Foundation forecasting models Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-15-foundation-models.qmd

## [Setup] Imports & Configuration --- Setup / Hidden in slides ---

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from nixtla import NixtlaClient
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae
plt.rcParams.update({"figure.figsize": (7, 3.5)})

## [Slide 5] 15.2 NHITS Transfer Learning Example

In [ ]:
from datasetsforecast.m4 import M4
from neuralforecast.utils import AirPassengersDF

Load M4 monthly data (source domain)

In [ ]:
Y_df = M4.load(directory="data", group="Monthly")[0].assign(
    ds=lambda df: df.groupby("unique_id")["ds"].transform(
        lambda x: pd.date_range(
            start="1970-01-01", periods=len(x), freq="MS"
        )
    )
)
horizon = 12
stacks = 3
models = [NHITS(
    input_size=5 * horizon, h=horizon,
    max_steps=2_000,
    stack_types=stacks * ["identity"],
    n_blocks=stacks * [1],
    mlp_units=[[256, 256] for _ in range(stacks)],
    n_pool_kernel_size=stacks * [1],
    batch_size=32, scaler_type="standard",
    n_freq_downsample=[12, 4, 1],
)]
nf = NeuralForecast(models=models, freq="MS")
nf.fit(df=Y_df)

Zero-shot on unseen Air Passengers (target domain)

In [ ]:
transfer_preds = nf.predict(df=AirPassengersDF.copy())

## [Slide 10] 15.3 Chronos (Amazon)

In [ ]:
from chronos import ChronosPipeline
import torch

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)
forecast = pipeline.predict(
    context=torch.tensor(y).unsqueeze(0),
    prediction_length=12,
)

## [Slide 13] 15.4 Example: Electricity Price Forecasting

In [ ]:
df = pd.read_csv(
    "data/electricity_short.csv", parse_dates=["ds"]
)

## [Slide 15] 15.4 TimeGPT Zero-Shot Forecasting

In [ ]:
nixtla_client = NixtlaClient(api_key="YOUR_API_KEY")

Zero-shot: no fine-tuning, direct application

In [ ]:
preds_df = nixtla_client.forecast(
    df=df, h=24, level=[80, 90]
)

Cross-validation for honest evaluation

In [ ]:
cv_preds_df = nixtla_client.cross_validation(
    df=df, h=24, n_windows=3
)

Evaluate

In [ ]:
evaluation = evaluate(
    cv_preds_df, models=["TimeGPT"], metrics=[mae]
)

## [Slide 17] 15.4 TimeGPT Fine-Tuning

In [ ]:
cv_finetune_preds_df = nixtla_client.cross_validation(
    df=df,
    h=24,
    n_windows=3,
    finetune_steps=15,    # number of gradient steps
    finetune_loss="mae",  # target metric to optimise
)

## [Slide 19] 15.4 TimeGPT with Exogenous Variables

In [ ]:
future_ex_vars_df = pd.read_csv(
    "data/electricity_future_vars.csv",
    parse_dates=["ds"],
)

Forecast with future exogenous variables

In [ ]:
exog_preds_df = nixtla_client.forecast(
    df=df,
    X_df=future_ex_vars_df,   # future covariates
    h=24,
    level=[80],
)

## [Slide 21] 15.4 Moirai Zero-Shot Forecasting

In [ ]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
import numpy as np

full_df = pd.concat([df, future_ex_vars_df], axis=0)
full_df = full_df.set_index("ds")
ds = PandasDataset.from_long_dataframe(
    full_df, target="y", item_id="unique_id",
    feat_dynamic_real=full_df.columns
        .drop(["unique_id", "y"]).tolist(),
)
train, test_template = split(ds, offset=-24)
test_data = test_template.generate_instances(
    prediction_length=24, windows=1, distance=24,
)
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(
        "Salesforce/moirai-1.0-R-large"
    ),
    prediction_length=24, context_length=240,
    patch_size="auto", num_samples=50,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)
predictor = model.create_predictor(batch_size=32)
forecasts = list(predictor.predict(test_data.input))

## [Slide 23] 15.4 Moirai Predictions and Combining Forecasts

In [ ]:
fc = forecasts[0]
moirai_preds_df = (
    pd.DataFrame(
        np.quantile(fc.samples, [0.5, 0.1, 0.9], axis=0).T
    )
    .set_axis(["Moirai", "Moirai-lo-80",
               "Moirai-hi-80"], axis="columns")
    .assign(ds=fc.index.to_timestamp())
)

Combine TimeGPT and Moirai forecasts

In [ ]:
fcst_df = preds_df.merge(exog_preds_df).merge(moirai_preds_df)

plot_series(
    df, fcst_df, max_insample_length=100,
    xlabel="Hour", ylabel="Price",
    title="Nord Pool electricity price",
    palette="black_and_3color", rm_legend=False,
    legend_loc="outside lower center",
)

## [Slide 25] 15.4 Chronos Usage

In [ ]:
from chronos import ChronosPipeline
import torch

Load pre-trained model (T5-small variant)

In [ ]:
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)

y: numpy array or list of historical values

In [ ]:
forecast = pipeline.predict(
    context=torch.tensor(y).unsqueeze(0),
    prediction_length=12,
    num_samples=20,    # number of sample paths
)

forecast shape: [num_series, num_samples, prediction_length]

In [ ]:
low, median, high = np.quantile(
    forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0
)